In [1]:
%pip install pandas

  Using cached pandas-2.3.3-cp310-cp310-macosx_10_9_x86_64.whl.metadata (91 kB)
  Using cached numpy-2.2.6-cp310-cp310-macosx_10_9_x86_64.whl.metadata (62 kB)
  Using cached pytz-2026.1.post1-py2.py3-none-any.whl.metadata (22 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 1.0 MB/s eta 0:00:0000:0100:010m
Using cached numpy-2.2.6-cp310-cp310-macosx_10_9_x86_64.whl (21.2 MB)

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [27]:
import sqlite3
import pandas as pd
import re

DB_FILE = "survey.db"
DUMP_FILE = "dump.sql"

conn = sqlite3.connect(DB_FILE)
cur = conn.cursor()

tables_sql = {
    "admins": """CREATE TABLE IF NOT EXISTS admins (
        id TEXT PRIMARY KEY,
        first_name TEXT,
        last_name TEXT,
        email TEXT,
        password TEXT,
        remember_token TEXT,
        created_at TEXT,
        updated_at TEXT
    )""",
    "answers": """CREATE TABLE IF NOT EXISTS answers (
        id TEXT PRIMARY KEY,
        text TEXT,
        variant_id TEXT,
        question_id TEXT,
        variant_value_id TEXT,
        user_data_id TEXT,
        theme_id TEXT,
        created_at TEXT,
        updated_at TEXT
    )""",
    "client_activity_logs": """CREATE TABLE IF NOT EXISTS client_activity_logs (
        id TEXT PRIMARY KEY,
        client_id TEXT,
        type TEXT,
        value TEXT,
        created_at TEXT,
        updated_at TEXT
    )""",
    "client_types": """CREATE TABLE IF NOT EXISTS client_types (
        id TEXT PRIMARY KEY,
        type TEXT,
        created_at TEXT,
        updated_at TEXT
    )""",
    "clients": """CREATE TABLE IF NOT EXISTS clients (
        id TEXT PRIMARY KEY,
        first_name TEXT,
        last_name TEXT,
        phone_number TEXT,
        organization TEXT,
        email TEXT,
        password TEXT,
        access_paid_content INTEGER,
        access_free_content INTEGER,
        client_type_id TEXT,
        created_at TEXT,
        updated_at TEXT,
        remember_token TEXT
    )""",
    "dynamic_supports": """CREATE TABLE IF NOT EXISTS dynamic_supports (
        id TEXT PRIMARY KEY,
        first_question_id TEXT,
        second_question_id TEXT,
        created_at TEXT,
        updated_at TEXT
    )""",
    "failed_jobs": """CREATE TABLE IF NOT EXISTS failed_jobs (
        id INTEGER PRIMARY KEY,
        uuid TEXT,
        connection TEXT,
        queue TEXT,
        payload TEXT,
        exception TEXT,
        failed_at TEXT
    )""",
    "faqs": """CREATE TABLE IF NOT EXISTS faqs (
        id TEXT PRIMARY KEY,
        question TEXT,
        answer TEXT,
        created_at TEXT,
        updated_at TEXT
    )""",
    "favorite_news": """CREATE TABLE IF NOT EXISTS favorite_news (
        id TEXT PRIMARY KEY,
        client_id TEXT,
        news_id TEXT,
        created_at TEXT,
        updated_at TEXT
    )""",
    "keywords": """CREATE TABLE IF NOT EXISTS keywords (
        id TEXT PRIMARY KEY,
        key TEXT,
        created_at TEXT,
        updated_at TEXT
    )""",
    "migrations": """CREATE TABLE IF NOT EXISTS migrations (
        id INTEGER PRIMARY KEY,
        migration TEXT,
        batch INTEGER
    )""",
    "model_has_permissions": """CREATE TABLE IF NOT EXISTS model_has_permissions (
        permission_id TEXT,
        model_type TEXT,
        model_id TEXT
    )""",
    "model_has_roles": """CREATE TABLE IF NOT EXISTS model_has_roles (
        role_id TEXT,
        model_type TEXT,
        model_id TEXT
    )""",
    "news": """CREATE TABLE IF NOT EXISTS news (
        id TEXT PRIMARY KEY,
        data TEXT,
        created_at TEXT,
        updated_at TEXT
    )""",
    "permissions": """CREATE TABLE IF NOT EXISTS permissions (
        uuid TEXT PRIMARY KEY,
        name TEXT,
        guard_name TEXT,
        created_at TEXT,
        updated_at TEXT
    )""",
    "questions": """CREATE TABLE IF NOT EXISTS questions (
        id TEXT PRIMARY KEY,
        title TEXT,
        short_title TEXT,
        tag TEXT,
        is_free INTEGER,
        is_cross INTEGER,
        is_correlation INTEGER,
        is_regression INTEGER,
        is_map_support INTEGER,
        keyword_id TEXT,
        theme_id TEXT,
        created_at TEXT,
        updated_at TEXT
    )""",
    "reports": """CREATE TABLE IF NOT EXISTS reports (
        id TEXT PRIMARY KEY,
        inputs TEXT,
        created_at TEXT,
        updated_at TEXT
    )""",
    "role_has_permissions": """CREATE TABLE IF NOT EXISTS role_has_permissions (
        permission_id TEXT,
        role_id TEXT
    )""",
    "roles": """CREATE TABLE IF NOT EXISTS roles (
        uuid TEXT PRIMARY KEY,
        name TEXT,
        guard_name TEXT,
        created_at TEXT,
        updated_at TEXT
    )""",
    "support_tickets": """CREATE TABLE IF NOT EXISTS support_tickets (
        id TEXT PRIMARY KEY,
        subject TEXT,
        message TEXT,
        phone_or_email TEXT,
        created_at TEXT,
        updated_at TEXT
    )""",
    "telescope_entries": """CREATE TABLE IF NOT EXISTS telescope_entries (
        sequence INTEGER PRIMARY KEY,
        uuid TEXT,
        batch_id TEXT,
        family_hash TEXT,
        should_display_on_index INTEGER,
        type TEXT,
        content TEXT,
        created_at TEXT
    )""",
    "telescope_entries_tags": """CREATE TABLE IF NOT EXISTS telescope_entries_tags (
        entry_uuid TEXT,
        tag TEXT
    )""",
    "telescope_monitoring": """CREATE TABLE IF NOT EXISTS telescope_monitoring (
        tag TEXT
    )""",
    "theme_files": """CREATE TABLE IF NOT EXISTS theme_files (
        id TEXT PRIMARY KEY,
        name TEXT,
        type TEXT,
        url TEXT,
        theme_id TEXT,
        created_at TEXT,
        updated_at TEXT
    )""",
    "themes": """CREATE TABLE IF NOT EXISTS themes (
        id TEXT PRIMARY KEY,
        title TEXT,
        year_id TEXT,
        data TEXT,
        is_free INTEGER,
        created_at TEXT,
        updated_at TEXT
    )""",
    "user_data": """CREATE TABLE IF NOT EXISTS user_data (
        id TEXT PRIMARY KEY,
        full_name TEXT,
        phone_number TEXT,
        region TEXT,
        age INTEGER,
        created_at TEXT,
        updated_at TEXT
    )""",
    "variant_values": """CREATE TABLE IF NOT EXISTS variant_values (
        id TEXT PRIMARY KEY,
        value INTEGER,
        variant_id TEXT,
        created_at TEXT,
        updated_at TEXT
    )""",
    "variants": """CREATE TABLE IF NOT EXISTS variants (
        id TEXT PRIMARY KEY,
        title TEXT,
        type TEXT,
        question_id TEXT,
        map_key TEXT,
        created_at TEXT,
        updated_at TEXT
    )""",
    "years": """CREATE TABLE IF NOT EXISTS years (
        id TEXT PRIMARY KEY,
        year INTEGER,
        created_at TEXT,
        updated_at TEXT
    )"""
}

for name, sql in tables_sql.items():
    cur.execute(sql)
conn.commit()

def parse_copy_block(lines):
    data = []
    for line in lines:
        if line.strip() == r"\.":
            break
        row = line.strip().split('\t')
        row = [None if v == '' else v for v in row]
        data.append(row)
    return data

with open(DUMP_FILE, "r", encoding="utf-8") as f:
    lines = f.readlines()

i = 0
while i < len(lines):
    line = lines[i]
    if line.startswith("COPY public."):
        match = re.match(r"COPY public.(\w+)", line)
        if not match:
            i += 1
            continue
        table_name = match.group(1)
        cols_match = re.search(r"\((.*?)\)", line)
        columns = [c.strip() for c in cols_match.group(1).split(',')]
        i += 1
        data = parse_copy_block(lines[i:])
        if data:
            df_temp = pd.DataFrame(data, columns=columns)
            df_temp.to_sql(table_name, conn, if_exists="append", index=False)
        i += len(data) + 1
    else:
        i += 1

query = """
SELECT
    t.id as theme_id,
    t.title as theme_title,
    tf.name as theme_file_name,
    tf.url as theme_file_url,
    q.id as question_id,
    q.title as question_title,
    u.id as user_id,
    u.full_name as user_name,
    u.phone_number as user_phone,
    u.age as user_age,
    u.region as user_region,
    a.id as answer_id,
    a.text as answer_text,
    v.title as variant_title,
    vv.value as variant_value,
    y.year as survey_year
FROM answers a
LEFT JOIN questions q ON a.question_id = q.id
LEFT JOIN themes t ON a.theme_id = t.id
LEFT JOIN theme_files tf ON tf.theme_id = t.id
LEFT JOIN user_data u ON a.user_data_id = u.id
LEFT JOIN variants v ON a.variant_id = v.id
LEFT JOIN variant_values vv ON a.variant_value_id = vv.id
LEFT JOIN years y ON t.year_id = y.id
"""

df = pd.read_sql(query, conn)
df.head()
conn.close()

                               theme_id  \
0  d9d6a391-0447-40f5-9fd4-a35a9dc2f32f   
1  d9d6a391-0447-40f5-9fd4-a35a9dc2f32f   
2  d9d6a391-0447-40f5-9fd4-a35a9dc2f32f   
3  d9d6a391-0447-40f5-9fd4-a35a9dc2f32f   
4  d9d6a391-0447-40f5-9fd4-a35a9dc2f32f   

                                         theme_title theme_file_name  \
0  Современные концептуальные подходы к содержани...            None   
1  Современные концептуальные подходы к содержани...            None   
2  Современные концептуальные подходы к содержани...            None   
3  Современные концептуальные подходы к содержани...            None   
4  Современные концептуальные подходы к содержани...            None   

  theme_file_url                           question_id  \
0           None  123e75d1-675c-4e51-be0e-3e1cdfee445a   
1           None  5563df2f-7320-4a30-9ef0-00b4402f75c5   
2           None  58aeff17-d2db-4a6c-b7c2-79472eeb7dd4   
3           None  7b9acf9f-d34f-4645-b77e-9bce6ee6b7aa   
4           None  

In [31]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("survey.db")

query = """
SELECT
    -- Тақырып
    t.id                AS theme_id,
    t.title             AS theme_title,
    t.is_free           AS theme_is_free,

    -- Жыл
    y.year              AS survey_year,

    -- Тақырып файлы
    tf.name             AS theme_file_name,
    tf.url              AS theme_file_url,

    -- Сұрақ
    q.id                AS question_id,
    q.title             AS question_title,
    q.short_title       AS question_short_title,
    q.tag               AS question_tag,

    -- Кілтсөз
    k.key               AS keyword,

    -- Нұсқа
    v.title             AS variant_title,
    v.type              AS variant_type,

    -- Нұсқа мәні
    vv.value            AS variant_value,

    -- Жауап
    a.id                AS answer_id,
    a.text              AS answer_text,

    -- Жауап беруші
    u.id                AS user_id,
    u.full_name         AS user_name,
    u.phone_number      AS user_phone,
    u.age               AS user_age,
    u.region            AS user_region

FROM answers a
LEFT JOIN questions      q   ON q.id  = a.question_id
LEFT JOIN themes         t   ON t.id  = a.theme_id
LEFT JOIN years          y   ON y.id  = t.year_id
LEFT JOIN theme_files    tf  ON tf.theme_id = t.id
LEFT JOIN keywords       k   ON k.id  = q.keyword_id
LEFT JOIN user_data      u   ON u.id  = a.user_data_id
LEFT JOIN variants       v   ON v.id  = a.variant_id
LEFT JOIN variant_values vv  ON vv.id = a.variant_value_id

ORDER BY
    survey_year,
    theme_title,
    question_title,
    u.full_name
"""

df = pd.read_sql_query(query, conn)

conn.close()

In [33]:
df.head()

,theme_id,theme_title,theme_is_free,survey_year,theme_file_name,theme_file_url,question_id,question_title,question_short_title,question_tag,...,variant_title,variant_type,variant_value,answer_id,answer_text,user_id,user_name,user_phone,user_age,user_region
0,37428318-4f02-4154-8625-e77d41c7aaec,Всемирное исследование ценностей в Казахстане ...,t,2011,None,None,3bb84b1c-6853-4b7f-9d2f-f6aa9b8ebd77,"Сіз қазір жұмыс істейсіз бе, жоқ па? Солай бо...",Жұмыс істейсіз бе?,v249,...,Аптасына 30 жəне одан көп сағат,radio,1,3393dbb5-a872-44cc-b3ad-9640d290f95f,\N,356bc8c0-9050-4fc3-a9b0-149b4d58082f,D11_1,D11_1,1,D11_1
1,37428318-4f02-4154-8625-e77d41c7aaec,Всемирное исследование ценностей в Казахстане ...,t,2011,None,None,3bb84b1c-6853-4b7f-9d2f-f6aa9b8ebd77,"Сіз қазір жұмыс істейсіз бе, жоқ па? Солай бо...",Жұмыс істейсіз бе?,v249,...,"Иə одан аз, аптасына 30 сағат",radio,2,2dfcb776-75cc-4881-87b9-4cd0ec8874f7,\N,597edd23-fd95-4e64-868a-9517db640522,D11_10,D11_10,10,D11_10
2,37428318-4f02-4154-8625-e77d41c7aaec,Всемирное исследование ценностей в Казахстане ...,t,2011,None,None,3bb84b1c-6853-4b7f-9d2f-f6aa9b8ebd77,"Сіз қазір жұмыс істейсіз бе, жоқ па? Солай бо...",Жұмыс істейсіз бе?,v249,...,"Иə одан аз, аптасына 30 сағат",radio,2,4dc646d5-7d72-4e06-9749-e675347c8a0a,\N,688667fa-e29c-4317-83f2-2ba87e2c4047,D11_100,D11_100,100,D11_100
3,37428318-4f02-4154-8625-e77d41c7aaec,Всемирное исследование ценностей в Казахстане ...,t,2011,None,None,3bb84b1c-6853-4b7f-9d2f-f6aa9b8ebd77,"Сіз қазір жұмыс істейсіз бе, жоқ па? Солай бо...",Жұмыс істейсіз бе?,v249,...,Аптасына 30 жəне одан көп сағат,radio,1,c5b73a82-a88d-4395-a418-6ab1275e3b65,\N,c6154340-80ab-4891-9989-26fb8a559b84,D11_1000,D11_1000,1000,D11_1000
4,37428318-4f02-4154-8625-e77d41c7aaec,Всемирное исследование ценностей в Казахстане ...,t,2011,None,None,3bb84b1c-6853-4b7f-9d2f-f6aa9b8ebd77,"Сіз қазір жұмыс істейсіз бе, жоқ па? Солай бо...",Жұмыс істейсіз бе?,v249,...,"Иə одан аз, аптасына 30 сағат",radio,2,8b0f8727-09ed-415c-9fd7-af7765d2dfef,\N,ee110466-394a-4033-846a-8c74d39196a2,D11_1001,D11_1001,1001,D11_1001


In [41]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("survey.db")

query = """
SELECT
    t.id                AS theme_id,
    t.title             AS theme_title,
    y.year              AS survey_year,
    q.id                AS question_id,
    q.title             AS question_title,
    q.short_title       AS question_short_title,
    q.tag               AS question_tag,
    k.key               AS keyword,
    u.id                AS user_id,
    u.full_name         AS user_name,
    u.phone_number      AS user_phone,
    u.age               AS user_age,
    u.region            AS user_region,
    a.text              AS answer_text,
    v.title             AS answer_variant

FROM answers a
LEFT JOIN questions      q   ON q.id  = a.question_id
LEFT JOIN themes         t   ON t.id  = a.theme_id
LEFT JOIN years          y   ON y.id  = t.year_id
LEFT JOIN keywords       k   ON k.id  = q.keyword_id
LEFT JOIN user_data      u   ON u.id  = a.user_data_id
LEFT JOIN variants       v   ON v.id  = a.variant_id
"""

df = pd.read_sql_query(query, conn)
conn.close()

df['answer_text'] = df['answer_text'].replace(r'\N', None, regex=False)
df['answer_final'] = df['answer_text'].fillna(df['answer_variant'])

df = df.drop(columns=['answer_text', 'answer_variant'])

df.to_json("survey_data.json", orient="records", force_ascii=False, indent=2)

print(f"Жазбалар саны: {len(df)}")
print("survey_data.json сақталды ✓")

Жазбалар саны: 2328525
survey_data.json сақталды ✓


In [ ]:
import pandas as pd

df = pd.read_json("survey_data.json")

for col in df.columns:
    print(f"{col}: {df[col].nunique()} уникалды мән")